# 02 — Feature Extraction Pipeline

This notebook walks through the full feature extraction pipeline step by step:

1. **Preprocessing** — load, normalize, and segment a raw audio clip.
2. **Log-Mel spectrogram** — convert the preprocessed clip to the CNN input representation.
3. **Audio-domain augmentation** — Gaussian noise and time shift visualised side-by-side with the original.
4. **Spectrogram-domain augmentation** — SpecAugment (frequency mask + time mask) visualised against the original.

All processing delegates to `src/preprocess.py` and `src/features.py`.
No pipeline logic is re-implemented here.

> **Pre-requisite:** `data/processed/` must contain at least one WAV file per class.
> Run `python src/preprocess.py --data-dir data/` first.

In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf

from src.config import CLASS_LABELS, DURATION, HOP_LENGTH, SAMPLE_RATE
from src.features import augment_audio, augment_spectrogram, compute_melspectrogram
from src.preprocess import load_audio, normalize, segment

DATA_DIR = ROOT / "data"
PROC_DIR = DATA_DIR / "processed"

print(f"ROOT     : {ROOT}")
print(f"PROC_DIR : {PROC_DIR}  (exists={PROC_DIR.exists()})")

In [ ]:
# Pick the first available WAV from the first class that has processed files.
sample_path  = None
sample_class = None
segmented    = None

for class_label in CLASS_LABELS.values():
    class_dir = PROC_DIR / class_label
    wavs = sorted(class_dir.glob("*.wav")) if class_dir.exists() else []
    if wavs:
        sample_path  = wavs[0]
        sample_class = class_label
        break

if sample_path is None:
    print("WARNING: No WAV files found in data/processed/")
    print("Run:  python src/preprocess.py --data-dir data/")
else:
    raw_audio = load_audio(sample_path, sr=SAMPLE_RATE)
    normed    = normalize(raw_audio)
    segmented = segment(normed, sr=SAMPLE_RATE, duration=DURATION)
    print(f"Loaded : {sample_path.name}  class={sample_class}")
    print(f"Shape  : {segmented.shape}  ({len(segmented)/SAMPLE_RATE:.1f} s @ {SAMPLE_RATE} Hz)")

## Section 1 — Preprocessing Pipeline

Three functions from `src/preprocess.py` standardise every clip before feature extraction:

| Step | Function | Effect |
|---|---|---|
| **Load** | `load_audio` | Resample to 22 050 Hz, return float32 mono array |
| **Normalize** | `normalize` | Scale so max\|x\| = 1.0; all-zero clips returned unchanged |
| **Segment** | `segment` | Center-crop if too long, symmetric zero-pad if too short → exactly 88 200 samples |

The same three functions are called by `online.py` at inference time, ensuring training
and live inference receive identical preprocessing.

In [ ]:
if segmented is None:
    print("No sample loaded — skipping preprocessing plot.")
else:
    t_raw = np.linspace(0, len(raw_audio) / SAMPLE_RATE, len(raw_audio))
    t_seg = np.linspace(0, DURATION, len(segmented))

    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=False)

    axes[0].plot(t_raw, raw_audio, linewidth=0.4, color="steelblue")
    axes[0].set_title(f"1. load_audio  —  {len(raw_audio)} samples")
    axes[0].set_ylabel("Amplitude")

    axes[1].plot(t_raw, normed, linewidth=0.4, color="darkorange")
    axes[1].set_title(f"2. normalize  —  max|x| = {np.max(np.abs(normed)):.4f}")
    axes[1].set_ylabel("Amplitude")

    axes[2].plot(t_seg, segmented, linewidth=0.4, color="seagreen")
    axes[2].set_title(
        f"3. segment  —  {len(segmented)} samples  "
        f"(target = {int(SAMPLE_RATE * DURATION)})"
    )
    axes[2].set_ylabel("Amplitude")
    axes[2].set_xlabel("Time (s)")

    fig.suptitle(f"Preprocessing steps — class: {sample_class}", fontsize=13)
    plt.tight_layout()
    plt.show()

## Section 2 — Log-Mel Spectrogram

`compute_melspectrogram` (from `src/features.py`) applies:

```python
mel_S = librosa.feature.melspectrogram(y=audio, sr=22050,
                                         n_fft=2048, hop_length=512, n_mels=128)
log_S = np.log(mel_S + 1e-6)   # shape: (128, 173)
```

The `+1e-6` floor prevents `log(0)` on silent frames.
The output is a **(128, 173)** float32 array — the direct input to the CNN after adding a channel
dimension: **(128, 173, 1)**.

In [ ]:
if segmented is None:
    print("No sample loaded — skipping spectrogram plot.")
else:
    log_S = compute_melspectrogram(segmented)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    t = np.linspace(0, DURATION, len(segmented))
    axes[0].plot(t, segmented, linewidth=0.4, color="steelblue")
    axes[0].set_title("Preprocessed waveform")
    axes[0].set_xlabel("Time (s)")
    axes[0].set_ylabel("Amplitude")

    img = librosa.display.specshow(
        log_S, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
        x_axis="time", y_axis="mel",
        ax=axes[1], cmap="magma",
    )
    axes[1].set_title(f"Log-Mel spectrogram  shape={log_S.shape}")
    fig.colorbar(img, ax=axes[1], label="Log energy")

    fig.suptitle(
        f"Waveform → log-Mel spectrogram — class: {sample_class}", fontsize=13
    )
    plt.tight_layout()
    plt.show()

    print(f"Shape         : {log_S.shape}   (n_mels × time_frames)")
    print(f"Value range   : [{log_S.min():.2f}, {log_S.max():.2f}]")
    print(f"Contains NaN  : {np.any(np.isnan(log_S))}")
    print(f"Contains Inf  : {np.any(np.isinf(log_S))}")

## Section 3 — Audio-domain Augmentation

`augment_audio(audio)` returns two waveform variants that are then converted to
spectrograms and saved alongside the original:

| Variant | Strategy | Parameter | Rationale |
|---|---|---|---|
| **Gaussian noise** | `audio + N(0, σ)` | σ = 0.005 | Simulates sensor or recording noise |
| **Time shift** | `np.roll(audio, shift)` | shift = ±10% of length | Teaches temporal invariance |

The left column shows the waveform; the right column shows the resulting spectrogram.
Compare rows to see how each augmentation changes the time-frequency representation.

In [ ]:
if segmented is None:
    print("No sample loaded — skipping audio augmentation plots.")
else:
    aug_audios = augment_audio(segmented)           # [noisy, shifted]
    aug_names  = ["Gaussian noise  (σ=0.005)", "Time shift  (±10%)"]
    aug_colors = ["tomato", "mediumpurple"]

    all_audio  = [segmented] + aug_audios
    all_names  = ["Original"] + aug_names
    all_colors = ["steelblue"] + aug_colors

    t = np.linspace(0, DURATION, len(segmented))

    fig, axes = plt.subplots(3, 2, figsize=(15, 10))

    for row, (audio_arr, name, color) in enumerate(
        zip(all_audio, all_names, all_colors)
    ):
        # Waveform column
        axes[row, 0].plot(t, audio_arr, linewidth=0.4, color=color)
        axes[row, 0].set_title(f"Waveform — {name}")
        axes[row, 0].set_ylabel("Amplitude")
        axes[row, 0].set_ylim(-1.15, 1.15)

        # Spectrogram column
        spec = compute_melspectrogram(audio_arr)
        img  = librosa.display.specshow(
            spec, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
            x_axis="time", y_axis="mel",
            ax=axes[row, 1], cmap="magma",
        )
        axes[row, 1].set_title(f"Spectrogram — {name}")
        fig.colorbar(img, ax=axes[row, 1], format="%+.1f")

    axes[-1, 0].set_xlabel("Time (s)")
    axes[-1, 1].set_xlabel("Time (s)")
    fig.suptitle(
        f"Audio-domain augmentations — class: {sample_class}", fontsize=13
    )
    plt.tight_layout()
    plt.show()

## Section 4 — Spectrogram-domain Augmentation (SpecAugment)

`augment_spectrogram(log_S)` masks the log-Mel matrix directly:

| Mask | Size | Rationale |
|---|---|---|
| **Frequency mask** | 20 Mel bins | Zeroes a random horizontal band — simulates partial frequency occlusion |
| **Time mask** | 30 frames (~0.7 s) | Zeroes a random vertical window — simulates signal dropout or clipping |

Both masks are applied together in a single call, returning one augmented spectrogram.
The original waveform is **unchanged** — only the spectrogram representation is masked.

In [ ]:
if segmented is None:
    print("No sample loaded — skipping SpecAugment plot.")
else:
    base_spec = compute_melspectrogram(segmented)
    spec_augs = augment_spectrogram(base_spec)  # list of 1

    fig, axes = plt.subplots(1, 1 + len(spec_augs), figsize=(14, 4))
    if len(spec_augs) == 0:
        axes = [axes]

    img0 = librosa.display.specshow(
        base_spec, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
        x_axis="time", y_axis="mel", ax=axes[0], cmap="magma",
    )
    axes[0].set_title("Original log-Mel spectrogram")
    fig.colorbar(img0, ax=axes[0], format="%+.1f", label="Log energy")

    for i, aug_spec in enumerate(spec_augs):
        img_a = librosa.display.specshow(
            aug_spec, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
            x_axis="time", y_axis="mel", ax=axes[i + 1], cmap="magma",
            vmin=base_spec.min(), vmax=base_spec.max(),
        )
        axes[i + 1].set_title(
            f"SpecAugment #{i + 1}\n"
            f"freq mask 20 bins  +  time mask 30 frames"
        )
        fig.colorbar(img_a, ax=axes[i + 1], format="%+.1f", label="Log energy")

    fig.suptitle(f"SpecAugment — class: {sample_class}", fontsize=13)
    plt.tight_layout()
    plt.show()

    zeroed = int(np.sum(spec_augs[0] == 0.0))
    print(f"Zeroed elements : {zeroed}  "
          f"({zeroed / spec_augs[0].size * 100:.1f}% of spectrogram)")

## Summary

The complete feature extraction pipeline is:

```
load_audio → normalize → segment
                ↓
    compute_melspectrogram          → (128, 173) log-Mel array
                ↓
    augment_audio (train only)      → [noisy, shifted] arrays → spectrograms
    augment_spectrogram (train only)→ [specaugmented] arrays
```

All four functions live in `src/preprocess.py` and `src/features.py`.
`dataset.py` enforces that augmented copies only appear in the training split.
`online.py` reuses `normalize`, `segment`, and `compute_melspectrogram` without
duplicating any logic.